In [1]:
import numpy as np
import pandas as pd

# 1. Load the dirty e-commerce dataset
# (Ensure this matches the exact filename unzipped in your folder)
df_dirty = pd.read_csv("orders.csv")

print("--- Initial Structural Review ---")
print(df_dirty.info())
print("\nFirst 5 Records:")
print(df_dirty.head())

print("\n--- Pipeline 1: Tracking Missing Value Footprints ---")
missing_counts = df_dirty.isnull().sum()
print(missing_counts[missing_counts > 0])

# 2. Executive Cleaning Action: Standardize dates and handle duplicates
print("\n--- Pipeline 2: Standardizing Timestamps & Dropping Duplicates ---")
# Strips whitespace and standardizes chaotic text layouts into true date formats
if "order_date" in df_dirty.columns:
    df_dirty["order_date"] = pd.to_datetime(
        df_dirty["order_date"].astype(str).str.strip(), errors="coerce"
    )

# FIX: Added [0] to extract the row count integer from the shape tuple
initial_rows = df_dirty.shape[0]
df_dirty.drop_duplicates(inplace=True)
clean_rows = df_dirty.shape[0]
print(f"Removed {initial_rows - clean_rows} exact duplicate entries.")

print("\n--- Pipeline 3: Validating Price/Financial Constraints ---")
# If cleaning order_items or products, check for pricing anomalies
if "price" in df_dirty.columns:
    invalid_prices = df_dirty[df_dirty["price"] <= 0].shape[0]
    print(f"Found {invalid_prices} records with zero or negative pricing metrics.")
    # Professional practice: Filter out invalid records
    df_dirty = df_dirty[df_dirty["price"] > 0]

# 3. Export Clean Baseline Dataset
df_dirty.to_csv("ecommerce_cleaned_baseline.csv", index=False)
print("\nCleaned baseline file successfully exported as 'ecommerce_cleaned_baseline.csv'!")

--- Initial Structural Review ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     80000 non-null  int64  
 1   customer_id  80000 non-null  int64  
 2   order_date   80000 non-null  object 
 3   ship_date    65547 non-null  object 
 4   status       80000 non-null  object 
 5   order_total  80000 non-null  float64
 6   country      79600 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 4.3+ MB
None

First 5 Records:
   order_id  customer_id  order_date   ship_date     status  order_total  \
0         1            1  2026-06-06         NaN    pending       109.48   
1         2            2  2025-11-17  2025-11-19  delivered        44.23   
2         3            2  2025-10-13         NaN  cancelled        41.39   
3         4            2  2026-08-01  2026-08-04  delivered        50.85   
4         5    

In [2]:
print("--- Pipeline 4: Advanced Conditional Inferences ---")

# 1. Cross-validate if missing ship_dates are due to Order Status
cancelled_pending_nan = df_dirty[df_dirty['status'].isin(['cancelled', 'pending'])]['ship_date'].isnull().sum()
total_nans = df_dirty['ship_date'].isnull().sum()

print(f"Total missing ship dates: {total_nans}")
print(f"Missing ship dates belonging to Cancelled or Pending orders: {cancelled_pending_nan}")

# Professional Action: Fill structural NaNs with a standard descriptor instead of leaving them blank
# This prevents ML algorithms or visualization tools from breaking later
df_dirty['ship_date'] = df_dirty['ship_date'].fillna('Not Shipped')


# 2. Handle missing Categorical values in 'country'
# Since it's a structural field, we will impute using the absolute mode (most frequent value)
country_mode = df_dirty['country'].mode()[0]
print(f"\nMost frequent country (Mode): {country_mode}")

print(f"Country missing values before imputation: {df_dirty['country'].isnull().sum()}")
df_dirty['country'] = df_dirty['country'].fillna(country_mode)
print(f"Country missing values after imputation: {df_dirty['country'].isnull().sum()}")


# 3. Final Integrity Audit Check
print("\n--- Final Integrity Verification Summary ---")
print(df_dirty.isnull().sum())

# Overwrite with fully processed dataset
df_dirty.to_csv("ecommerce_final_clean.csv", index=False)
print("\nFully processed production dataset saved as 'ecommerce_final_clean.csv'!")


--- Pipeline 4: Advanced Conditional Inferences ---
Total missing ship dates: 14453
Missing ship dates belonging to Cancelled or Pending orders: 14104

Most frequent country (Mode): United States
Country missing values before imputation: 400
Country missing values after imputation: 0

--- Final Integrity Verification Summary ---
order_id       0
customer_id    0
order_date     0
ship_date      0
status         0
order_total    0
country        0
dtype: int64

Fully processed production dataset saved as 'ecommerce_final_clean.csv'!
